In [1]:
import os
os.environ['HF_HOME'] = '/projectnb/vkolagrp/skowshik/.cache/'
import torch
import argparse
import pandas as pd
import torch.distributed as dist
import json
import warnings
import random
warnings.filterwarnings("ignore")

from tqdm import tqdm
from datetime import timedelta
from transformers import AutoTokenizer, AutoModelForCausalLM
from vllm import LLM, SamplingParams

In [2]:
data_path = "../../data/1022/csv_to_txt/all_nacc_csv_to_txt.csv"
datadict_path = "/projectnb/vkolagrp/datasets/NACC/data_dictionaries/NACC_dictionary.json"
save_dir = "../../data/1115"
json_name = f"{save_dir}/test.json"
max_new_tokens = 1024
batch_size = 512
model_id = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
n_devices = 1
n_cases = 1000

In [9]:
def create_batches(dataframe, batch_size):
    """Yield successive batch_size-sized batches from dataframe

    Args:
        dataframe: pandas dataframe object
        batch_size: batch size

    Returns:
        Yields batch_size sized batches from dataframe
    """
    for i in range(0, len(dataframe), batch_size):
        yield dataframe.iloc[i:i + batch_size]
        
def load_model(model_id):
    """Load VLLM model and Huggingface tokenizer."""
    print(model_id)
    llm = LLM(
        model=model_id,
        tokenizer=model_id,
        tensor_parallel_size=n_devices,
        gpu_memory_utilization=0.90,
        # max_model_len=30000,
        enable_lora=False,
        distributed_executor_backend='ray'
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side='left')
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    return llm, tokenizer

def load_json(dataset_path):
    """
    Load the json file
    :param dataset_path: the path to the json file
    :return data: data from json
    """
    try:
        with open(dataset_path, 'r', encoding='utf-8') as json_file:
            data = json.load(json_file)
    except:
        data = []
        with open(dataset_path, "r", encoding='utf-8') as json_file:
            for line in json_file:
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON: {e}")

    return data

def load_data(data_path):
    """Load the dataset."""
    data = pd.read_csv(data_path)
    data = data[data['NACCUDSD'].isin([1,3,4])][:n_cases]
    print(f"Number of cases to process: {len(data)}")
    return data

def get_diagnosis_dict(datadict_path):
    """Load the NACC data dictionary."""
    datadict = load_json(datadict_path)
                
    diagnosis_dict = {'Cognitive Status': datadict['NACCUDSD']['Values']} # Can add more diagnosis variables to this dictionary
    diagnosis_dict['Cognitive Status'].pop('2')
    
    return diagnosis_dict

In [5]:
def get_summary_llama(llm, tokenizer, input_texts, system_msg):
    """This is a function to generate LLAMA summaries using vllm https://github.com/vllm-project/vllm

    Args:
        llm: LLM object
        tokenizer: Huggingface tokenizer
        input_texts (List): A list of input texts / prompts
        system_msg (str): system message for the LLAMA prompt

    Returns:
        List: A list of generated responses
    """
    
    prompts = []
    responses = []
    
    messages = [
        [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{text['instruction']}\n\n### Input:\n{text['user']}"},
        ] for text in input_texts
    ]
    # print(len(messages))
    # raise ValueError
    
    for message in messages:
        input_ids = tokenizer.apply_chat_template(
            message,
            add_generation_prompt=True,
            tokenize=False,
            # return_tensors="pt"
        )
        
        prompts.append(input_ids)
        
    # stop_tokens = stop_token_list()
    
    # https://github.com/vllm-project/vllm/blob/main/vllm/sampling_params.py#L38-L66
    sampling_params = SamplingParams(
        temperature=0.1,
        max_tokens=max_new_tokens,
        frequency_penalty=0.5,
        # stop=stop_tokens
    )
    
    completions = llm.generate(
        prompts=prompts,
        sampling_params=sampling_params,
    )
    
    for i, output in enumerate(completions):
        temp_gen = output.outputs[0].text
        responses.append(temp_gen)
        
    print('Successfully finished generating', len(prompts), 'samples!')
    
    return responses

In [6]:
def generate_data(json_name, data, questions, llm, tokenizer, system_msg):
    """This is a function to generate LLAMA summaries using vllm and save the generated responses to a json file. 
    NOTE: The responses are saved as a list of dictionaries instead of a nested dictionary. Need to change this later.

    Args:
        json_name: name of the json file to save the results
        data: pandas dataframe object
        questions: list of questions/instructions
        llm: LLM object
        tokenizer: Huggingface tokenizer
        system_msg (str): system message for the LLAMA prompt
    """
    for batch in tqdm(create_batches(data, batch_size)):
        patient_texts = batch['patient_summary'].tolist()
        
        instructions = []
        for i, question in enumerate(questions):
            
            modified_patient_texts = [
                {'instruction': f"{question}", 'user': f"Patient:{patient_text}"} for patient_text in enumerate(patient_texts)
            ]
            
            responses = get_summary_llama(llm, tokenizer, modified_patient_texts, system_msg=system_msg)
            
            # Modify this according to what data you want to save in the json file along with the llama generated answers
            print(f"Generating diagnosis for question: {question}")
            for i, row in batch.iterrows():
                with open(json_name, 'a', encoding='utf-8') as json_file:
                    summary_data = { # Add the fields that are necessary to identify a specific NACC case
                        "index": row.name,
                        "NACCID": row["NACCID"],
                        "NACCVNUM": row["NACCVNUM"],
                        "NACCMNUM": row["NACCMNUM"],
                        "NACCNMRI": row["NACCNMRI"],
                        "NACCMRSA": row["NACCMRSA"],
                        "NACCADC": row["NACCADC"],
                        "VISITMO": row["VISITMO"],
                        "VISITDAY": row["VISITDAY"],
                        "VISITYR": row["VISITYR"],
                        "MRIMO": row["MRIMO"],
                        "MRIDY": row["MRIDY"],
                        "MRIYR": row["MRIYR"],
                        "NACCUDSD": row["NACCUDSD"],
                        "NACCETPR": row["NACCETPR"],
                        "CDRGLOB": row["CDRGLOB"],
                        "instruction": question,
                        "patient_summary": row["patient_summary"],
                        "diag_summary": row["diag_summary"],
                        "answer": responses[i - batch.first_valid_index()]
                    }
                    json_file.write(json.dumps(summary_data, ensure_ascii=False) + "\n")
                    json_file.close()
        
        torch.cuda.empty_cache()

## Load model

In [7]:
llm, tokenizer = load_model(model_id)

meta-llama/Meta-Llama-3.1-8B-Instruct
WARNING 11-15 11:00:59 arg_utils.py:957] Chunked prefill is enabled by default for models with max_model_len > 32K. Currently, chunked prefill might not work with some features or models. If you encounter any issues, please disable chunked prefill by setting --enable-chunked-prefill=False.
INFO 11-15 11:00:59 config.py:1021] Chunked prefill is enabled with max_num_batched_tokens=512.


2024-11-15 11:01:00,573	INFO worker.py:1786 -- Started a local Ray instance.


INFO 11-15 11:01:01 llm_engine.py:237] Initializing an LLM engine (v0.6.3.post1) with config: model='meta-llama/Meta-Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Meta-Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=meta-llama/Meta-Llama-3.1-8B-Instruct, num_scheduler_steps=1, chunked_prefill_enabled=True

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 11-15 11:01:09 model_runner.py:1067] Loading model weights took 14.9888 GB
INFO 11-15 11:01:09 distributed_gpu_executor.py:57] # GPU blocks: 12114, # CPU blocks: 2048
INFO 11-15 11:01:09 distributed_gpu_executor.py:61] Maximum concurrency for 131072 tokens per request: 1.48x
INFO 11-15 11:01:10 model_runner.py:1395] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 11-15 11:01:10 model_runner.py:1399] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 11-15 11:01:21 model_runner.py:1523] Graph capturing finished in 10 secs.


## Load data

In [10]:
print(json_name)
data = load_data(data_path)
if not os.path.exists(save_dir):
    os.makedirs(save_dir, exist_ok=True)

# Load the diagnosis dictionary
diagnosis_dict = get_diagnosis_dict(datadict_path)

../../data/1115/test.json
Number of cases to process: 1000


## Run VLLM inference

In [11]:
questions = [
        """Provide the Cognitive status at UDS visit for a patient presenting with the following information. Data dictionary:\n"""+ json.dumps(diagnosis_dict, indent=4), 
    ]
    
# System message
system_msg = """You are a helpful behavioral neurologist AI assisstant. You will be given patient information, an instruction and a data dictionary. Your purpose is to answer the given question, and report them in JSON format according to the provided dictionary.
Print your response following the JSON template
{
    "Answer" (Replace this word according the asked question): {
        "value": ...,
        "explanation": ...
    }
}
Do not print anything else, only the JSON. Any explanation must be in the 'explanation' key. Provide a complete and elaborate explanation.
"""

In [12]:
generate_data(json_name=json_name, data=data, questions=questions, llm=llm, tokenizer=tokenizer, system_msg=system_msg)

0it [00:00, ?it/s]

## Load generated summaries

In [5]:
generated_responses = load_json(json_name)

In [11]:
correct_diag = 0
for i, entry in enumerate(generated_responses):
    json_entry = json.loads(entry['answer'])
    if json_entry['Cognitive Status']['value'] == entry['NACCUDSD']:
        correct_diag += 1

In [12]:
accuracy = correct_diag / len(generated_responses)
print(f"Cognitive status accuracy: {accuracy}")

Cognitive status accuracy: 0.695
